# Week 12 Homework

## Joseph Cornell

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Seed the pseudo-random number generator
np.random.seed(12)

# Sample size
n = 100

# Create data
wcc = np.round(np.random.normal(15, 5, n), 0)
crp = (wcc * 2) + np.round(np.random.normal(0, 10, n), 0)
lipase = wcc + crp + np.round(np.random.normal(2, 10, n), 0)

# Pandas dataframe object
df = pd.DataFrame({
    'WCC': wcc,
    'CRP': crp,
    'Lipase': lipase
})

# Display first few rows
print("Data Summary:")
print(df.head(10))
print(f"\nDataset shape: {df.shape}")
print(f"\nData statistics:")
print(df.describe())

In [ ]:
# Question 1: Create multiple linear regression model with WCC and CRP as predictors of Lipase
X = df[['WCC', 'CRP']]
y = df['Lipase']

# Create and fit the model
model = LinearRegression()
model.fit(X, y)

# Get predictions
y_pred = model.predict(X)

# Calculate R-squared
r_squared = model.score(X, y)

# Calculate residuals
residuals = y - y_pred

# Calculate Mean Squared Error and Root Mean Squared Error
mse = np.mean(residuals**2)
rmse = np.sqrt(mse)

# Calculate F-statistic manually
ss_total = np.sum((y - np.mean(y))**2)
ss_residual = np.sum(residuals**2)
p = X.shape[1]  # number of predictors
f_statistic = ((ss_total - ss_residual) / p) / (ss_residual / (n - p - 1))

# Calculate p-value for F-statistic
f_pvalue = 1 - stats.f.cdf(f_statistic, p, n - p - 1)

# Print regression results
print("="*60)
print("MULTIPLE LINEAR REGRESSION RESULTS")
print("="*60)
print(f"\nModel: Lipase ~ WCC + CRP")
print(f"\nCoefficients:")
print(f"  Intercept: {model.intercept_:.4f}")
for i, col in enumerate(X.columns):
    print(f"  {col}: {model.coef_[i]:.4f}")

print(f"\nModel Fit Statistics:")
print(f"  R-squared: {r_squared:.4f}")
print(f"  Adjusted R-squared: {1 - (1 - r_squared) * (n - 1) / (n - p - 1):.4f}")
print(f"  F-statistic: {f_statistic:.4f}")
print(f"  F p-value: {f_pvalue:.4e}")
print(f"  RMSE: {rmse:.4f}")

# Calculate t-statistics and p-values for coefficients
X_with_const = np.column_stack([np.ones(n), X])
XtX_inv = np.linalg.inv(X_with_const.T @ X_with_const)
mse_actual = ss_residual / (n - p - 1)
se = np.sqrt(np.diag(XtX_inv) * mse_actual)

# Calculate t-statistics
t_stats = np.concatenate([[model.intercept_], model.coef_]) / se
p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), n - p - 1))

print(f"\nCoefficient Statistics:")
print(f"  Intercept: t = {t_stats[0]:.4f}, p-value = {p_values[0]:.4f}")
for i, col in enumerate(X.columns):
    print(f"  {col}: t = {t_stats[i+1]:.4f}, p-value = {p_values[i+1]:.4f}")
print("="*60)

## Question 2: Comment on the F statistic

The F-statistic tests the overall significance of the regression model, evaluating whether at least one predictor variable has a non-zero coefficient.

**Results:**
- **F-statistic = 157.2870**
- **p-value < 0.0001** (essentially zero)

**Interpretation:**
Since the p-value is far less than 0.05, we reject the null hypothesis that all slope coefficients equal zero.

**Conclusion:** The overall regression model is **highly statistically significant**. At least one (and in this case, both) of the predictor variables (WCC or CRP) has a significant linear relationship with Lipase levels. This strong F-statistic indicates that the model as a whole provides a better fit to the data than simply using the mean of Lipase as the predictor.

## Question 3: Comment on the individual coefficients and their p-values

**WCC Coefficient Analysis:**
- **Coefficient: 0.5426** (the slope of WCC)
- t-statistic: 2.0213
- **p-value: 0.0460** (statistically significant at α = 0.05)
- **Interpretation:** WCC is a statistically significant predictor of Lipase (p < 0.05). For each unit increase in WCC (holding CRP constant), Lipase is expected to increase by approximately 0.54 units.

**CRP Coefficient Analysis:**
- **Coefficient: 1.1827** (the slope of CRP)
- t-statistic: 11.6636
- **p-value: 0.0000** (highly significant)
- **Interpretation:** CRP is a highly statistically significant predictor of Lipase (p < 0.0001). For each unit increase in CRP (holding WCC constant), Lipase is expected to increase by approximately 1.18 units. CRP has a stronger effect than WCC.

**Intercept:**
- **Coefficient: 2.3741**
- t-statistic: 0.7880
- p-value: 0.4326 (not statistically significant)
- **Interpretation:** The intercept is not significantly different from zero, meaning the model predicts Lipase ≈ 2.37 when both WCC and CRP are zero.

**Overall Conclusion:** CRP is the more influential predictor, but both predictors contribute meaningfully to explaining Lipase variation. The individual p-values show that while WCC is marginally significant, CRP is highly significant.

## Question 4: Comment on the R² value

**R-squared (R²) Analysis:**
- **R² value: 0.7643**
- This means that **76.43%** of the variance in Lipase levels is explained by the combination of WCC and CRP

**Interpretation:**
- R² ranges from 0 to 1, where 1 represents a perfect fit
- An R² of **0.7643 indicates a STRONG relationship**:
  - If R² > 0.7: **Strong relationship** ← We're here
  - If R² between 0.4-0.7: Moderate relationship
  - If R² < 0.4: Weak relationship

**What This Means:**
- This indicates a **strong model fit**. The two predictors explain more than three-quarters of the variation in Lipase levels
- The remaining **23.57%** of variance could be due to:
  - Unmeasured biological variables
  - Measurement error
  - Inherent biological variability
  - Other factors not included in this model

**Adjusted R² Consideration:**
- **Adjusted R² = 0.7595** (slightly lower than R² due to the number of predictors)
- This adjusted value accounts for overfitting and is often more reliable when comparing models with different numbers of predictors
- The small difference between R² and Adjusted R² suggests we're not overfitting the data

**Conclusion:** The model has excellent explanatory power, with both predictors explaining most of the variation in Lipase levels.

## Summary

This multiple linear regression analysis examined the relationship between WCC, CRP, and Lipase levels using 100 observations.

**Key Findings:**
1. **Model Significance**: The regression model is highly significant (F = 157.29, p < 0.0001)
2. **Predictor Quality**: Both WCC (p = 0.046) and CRP (p < 0.0001) are significant predictors
3. **Model Fit**: The model explains 76.43% of the variance in Lipase (R² = 0.7643)
4. **Relative Importance**: CRP is the stronger predictor (coefficient = 1.1827 vs WCC = 0.5426)

This analysis demonstrates that white blood cell count and C-reactive protein together provide strong predictive power for lipase levels in this dataset.